In [9]:
import os
import torch
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    SegformerImageProcessor, 
    SegformerForSemanticSegmentation, 
    TrainingArguments, 
    Trainer
)

In [ ]:

IMAGE_DIR = "./data/JPEGImages"
MASK_DIR = "./data/SegmentationClass"
MODEL_ID = "nvidia/mit-b0"  # Small, fast, and accurate for most tasks

ID2LABEL = {0: "background", 1: "filth"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

In [11]:
# Get all image filenames
all_filenames = sorted([
    f for f in os.listdir(IMAGE_DIR) 
    if f.lower().endswith(('.jpg', '.png', '.jpeg'))
])

# 80/20 Split
train_files, val_files = train_test_split(
    all_filenames, 
    test_size=0.2, 
    random_state=42, 
    shuffle=True
)

print(f"Total images: {len(all_filenames)}")
print(f"Training on: {len(train_files)}")
print(f"Validating on: {len(val_files)}")

Total images: 99
Training on: 79
Validating on: 20


In [12]:
def run_sanity_check(filenames, img_dir, mask_dir):
    print(f"Starting check on {len(filenames)} files...")
    issues = 0
    
    for fname in filenames:
        img_path = os.path.join(img_dir, fname)
        mask_name = os.path.splitext(fname)[0] + ".png"
        mask_path = os.path.join(mask_dir, mask_name)
        
        # 1. Check if files exist
        if not os.path.exists(img_path):
            print(f"MISSING IMAGE: {img_path}")
            issues += 1
            continue
        if not os.path.exists(mask_path):
            print(f"MISSING MASK: {mask_path}")
            issues += 1
            continue
            
        try:
            img = Image.open(img_path)
            mask = Image.open(mask_path)
            
            # 2. Check if dimensions match
            if img.size != mask.size:
                print(f"SIZE MISMATCH: {fname} (Img: {img.size}, Mask: {mask.size})")
                issues += 1
            
            # 3. Check mask values (Ensure they aren't all zero)
            mask_array = np.array(mask)
            unique_vals = np.unique(mask_array)
            if len(unique_vals) <= 1:
                # Warning only: some images might actually only have background
                pass 
                
        except Exception as e:
            print(f"CORRUPT FILE: {fname} - {e}")
            issues += 1

    if issues == 0:
        print("✅ Sanity check passed! Your dataset is ready for training.")
    else:
        print(f"❌ Found {issues} issues. Please fix them before training.")

# Run it
run_sanity_check(all_filenames, IMAGE_DIR, MASK_DIR)

Starting check on 99 files...
✅ Sanity check passed! Your dataset is ready for training.


In [17]:
class SegformerDataset(Dataset):
    def __init__(self, filenames, img_dir, mask_dir, processor):
        self.filenames = filenames
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.processor = processor

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        
        image = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        mask_name = os.path.splitext(fname)[0] + ".png"
        mask = Image.open(os.path.join(self.mask_dir, mask_name)).convert("L")

        # --- FIX START: SANITIZE MASK VALUES ---
        mask_array = np.array(mask)
        
        # Option A: If your data is just Background vs Filth, 
        # make everything that isn't 0 into 1.
        mask_array[mask_array > 0] = 1
        
        # Option B (Use this if you have noisy edges): 
        # Only treat strong signals (e.g. > 128) as filth, ignore weak noise
        # mask_array = (mask_array > 128).astype(np.uint8)
        
        # Convert back to PIL for the processor
        mask = Image.fromarray(mask_array)
        # --- FIX END ---

        encoded_inputs = self.processor(
            image, 
            mask, 
            return_tensors="pt"
        )
        
        for k, v in encoded_inputs.items():
            encoded_inputs[k] = v.squeeze(0)
            
        return encoded_inputs

In [18]:
processor = SegformerImageProcessor.from_pretrained(MODEL_ID)
processor.do_reduce_labels = False 

train_dataset = SegformerDataset(train_files, IMAGE_DIR, MASK_DIR, processor)
val_dataset = SegformerDataset(val_files, IMAGE_DIR, MASK_DIR, processor)

model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_ID,
    num_labels=len(ID2LABEL),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

In [ ]:
device = torch.device("cuda")
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Usage: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_ID,
    num_labels=len(ID2LABEL),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

model.to(device)

training_args = TrainingArguments(
    output_dir="./segformer-finetuned-results",
    learning_rate=6e-5,
    num_train_epochs=50,
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=10,
    load_best_model_at_end=True,
    remove_unused_columns=False,
    fp16=True, 
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

Using device: cuda


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx